## SelfGNN — Finance Feature Extraction (4+4 Schema)

**Prerequisite:** Run `preprocess_finance.ipynb` first.

| Side | Dim | Features | Rationale |
|------|-----|----------|-----------|
| User | 4 | `avg_interaction_value`, `activity_span_days`, `recency_score`, `value_std_norm` | Non-graph-redundant signals |
| Merchant | 4 | `avg_interaction_value`, `category_id` (MCC index), `value_std_norm`, `user_repeat_rate` | Non-graph-redundant signals |
| Edge | 1 | `normalized_interaction_value` (sigmoid(log1p(|amount|) / p75)) | |

**Dropped (graph-redundant):** `interaction_count`, `unique_*_count`, `txns_per_user`, `popularity_rank` — all derivable from graph degree/neighbourhood; GNN already encodes these implicitly.

**Outputs:** `Datasets/finance-merchant/`
- `user_features.npy` — shape (N_users, 4)
- `merchant_features.npy` — shape (N_merchants, 4)
- `edge_weights.pkl`, `feature_meta.json`

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUT_DIR      = '../Datasets/finance-merchant/'
RAW_TXN      = '../datasetRaw/finance/transactions_data.csv'
DATASET_NAME = 'finance-merchant'

import os, json, pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime

with open(OUT_DIR + 'user2id.pkl', 'rb') as f:
    user2id = pickle.load(f)
with open(OUT_DIR + 'merchant2id.pkl', 'rb') as f:
    merchant2id = pickle.load(f)

usrnum      = len(user2id)
merchantnum = len(merchant2id)
print(f'Loaded mappings: {usrnum:,} users, {merchantnum:,} merchants')

# ── Load the 2-year window from preprocess_finance.ipynb ──────────────────────
with open(OUT_DIR + 'window_config.json') as f:
    _win = json.load(f)
CUTOFF_TS    = int(_win['cutoff_ts'])
MAX_TS       = int(_win['max_ts'])
LAST_N_YEARS = int(_win['last_n_years'])
print(f'Window       : {datetime.fromtimestamp(CUTOFF_TS)} → '
      f'{datetime.fromtimestamp(MAX_TS)}  ({LAST_N_YEARS} yrs)')

Loaded mappings: 1,210 users, 17,377 merchants
Window       : 2017-11-01 07:59:00 → 2019-11-01 07:59:00  (2 yrs)


In [2]:
# ── Cell 2: Pre-scan to compute value-normalisation reference ─────────────────
# Collects per-transaction log1p(|amount|) within the 2-year window so we can
# later map values to (0,1) for the NODE `avg_interaction_value` feature. Edge
# weights use a separate rank-percentile normalisation in Cell 6.
all_amounts = []
for chunk in pd.read_csv(RAW_TXN, chunksize=500_000, low_memory=False,
                          usecols=['client_id', 'merchant_id', 'date', 'amount', 'mcc']):
    chunk['amount']      = pd.to_numeric(
        chunk['amount'].astype(str).str.replace('$', '', regex=False), errors='coerce')
    chunk['client_id']   = chunk['client_id'].astype(str)
    chunk['merchant_id'] = chunk['merchant_id'].astype(str)
    chunk['date']        = pd.to_datetime(chunk['date'], errors='coerce')
    chunk = chunk[
        chunk['client_id'].isin(user2id) &
        chunk['merchant_id'].isin(merchant2id)
    ].dropna(subset=['amount', 'date'])
    chunk['unix_ts']     = chunk['date'].astype(np.int64) // 10**9
    chunk = chunk[chunk['unix_ts'] >= CUTOFF_TS]  # 2-year window
    all_amounts.append(chunk['amount'].abs().values)

amounts_flat = np.concatenate(all_amounts)
log_amounts  = np.log1p(amounts_flat)
p75          = float(max(np.percentile(log_amounts, 75), 1.0))
print(f'Amount p75 (log scale, window-filtered): {p75:.4f}')

def norm_amount(amt_abs: np.ndarray) -> np.ndarray:
    """sigmoid(log1p(|amount|) / p75) → (0, 1). Used for NODE value stats; edge
    weights use rank-percentile instead."""
    return 1.0 / (1.0 + np.exp(-np.log1p(amt_abs) / p75))

Amount p75 (log scale, window-filtered): 4.2627


In [3]:
# ── Cell 3: Build per-user/merchant/edge statistics ────────────────────────────
u_count     = np.zeros(usrnum,      dtype=np.int64)
u_val_sum   = np.zeros(usrnum,      dtype=np.float64)
u_val_sq    = np.zeros(usrnum,      dtype=np.float64)
u_merchants = [set() for _ in range(usrnum)]
u_min_ts    = np.full(usrnum,  np.inf)
u_max_ts    = np.full(usrnum, -np.inf)

m_count     = np.zeros(merchantnum, dtype=np.int64)
m_val_sum   = np.zeros(merchantnum, dtype=np.float64)
m_val_sq    = np.zeros(merchantnum, dtype=np.float64)
m_users     = [set() for _ in range(merchantnum)]
m_mcc_seen: dict = {}

edge_val_sum: dict = defaultdict(float)
edge_cnt:     dict = defaultdict(int)
global_max_ts      = -np.inf

edge_amt_sum: dict = defaultdict(float)   # for rank-percentile edge weights

for chunk in pd.read_csv(RAW_TXN, chunksize=500_000, low_memory=False,
                          usecols=['client_id', 'merchant_id', 'date', 'amount', 'mcc']):
    chunk['amount']      = pd.to_numeric(
        chunk['amount'].astype(str).str.replace('$', '', regex=False), errors='coerce')
    chunk['client_id']   = chunk['client_id'].astype(str)
    chunk['merchant_id'] = chunk['merchant_id'].astype(str)
    chunk['date']        = pd.to_datetime(chunk['date'], errors='coerce')
    chunk = chunk[
        chunk['client_id'].isin(user2id) &
        chunk['merchant_id'].isin(merchant2id)
    ].dropna(subset=['amount', 'date'])

    chunk['uid']      = chunk['client_id'].map(user2id)
    chunk['mid']      = chunk['merchant_id'].map(merchant2id)
    chunk['norm_val'] = norm_amount(chunk['amount'].abs().values)
    chunk['unix_ts']  = chunk['date'].astype(np.int64) // 10**9
    chunk             = chunk[chunk['unix_ts'] >= CUTOFF_TS]  # 2-year window
    chunk['abs_amt']  = chunk['amount'].abs()

    for row in chunk[['uid', 'mid', 'norm_val', 'unix_ts', 'mcc', 'abs_amt']].itertuples(index=False):
        uid, mid = int(row.uid), int(row.mid)
        v = float(row.norm_val)
        ts = int(row.unix_ts)

        u_count[uid]   += 1
        u_val_sum[uid]  += v
        u_val_sq[uid]   += v * v
        u_merchants[uid].add(mid)
        if ts < u_min_ts[uid]: u_min_ts[uid] = ts
        if ts > u_max_ts[uid]: u_max_ts[uid] = ts
        if ts > global_max_ts: global_max_ts = ts

        m_count[mid]   += 1
        m_val_sum[mid]  += v
        m_val_sq[mid]   += v * v
        m_users[mid].add(uid)

        mcc_val = row.mcc
        if pd.notna(mcc_val) and mid not in m_mcc_seen:
            m_mcc_seen[mid] = int(mcc_val)

        edge_val_sum[(uid, mid)] += v
        edge_cnt[(uid, mid)]     += 1
        edge_amt_sum[(uid, mid)] += float(row.abs_amt)

print(f'Total transactions in scope: {u_count.sum():,}')

Total transactions in scope: 2,749,810


In [4]:
# ── Cell 4: Build 4+4 feature arrays ──────────────────────────────────────────
# Kept  (user)    : avg_interaction_value, activity_span_days, recency_score, value_std_norm
# Kept  (merchant): avg_interaction_value, category_id (MCC), value_std_norm, user_repeat_rate
# Dropped: interaction_count, unique_*_count, txns_per_week, repeat_visit_rate, popularity_rank
# Rationale: dropped features are graph-redundant (= node degree, neighbourhood size, or
# algebraic derivatives thereof). GNN propagation already encodes them implicitly.
# Kept features provide orthogonal signal: interaction quality (avg_value), temporal
# profile (span / recency), preference volatility (value_std), and the only exogenous
# signal available (MCC merchant category).

SECS_PER_DAY  = 86400.0
SECS_PER_YEAR = 365.0 * SECS_PER_DAY

u_avg_val       = np.where(u_count > 0, u_val_sum / u_count, 0.0).astype(np.float32)
u_var           = np.maximum(u_val_sq / np.maximum(u_count, 1) - (u_val_sum / np.maximum(u_count, 1))**2, 0.0)
u_val_std       = np.sqrt(u_var).astype(np.float32)
u_unique_merch  = np.array([len(s) for s in u_merchants], dtype=np.float32)
u_span_days     = np.where(np.isfinite(u_min_ts) & np.isfinite(u_max_ts),
                            (u_max_ts - u_min_ts) / SECS_PER_DAY, 0.0).astype(np.float32)
u_recency       = np.where(np.isfinite(u_max_ts),
                            np.exp(-np.maximum(0.0, global_max_ts - u_max_ts) / SECS_PER_YEAR),
                            0.0).astype(np.float32)

m_avg_val       = np.where(m_count > 0, m_val_sum / m_count, 0.0).astype(np.float32)
m_var           = np.maximum(m_val_sq / np.maximum(m_count, 1) - (m_val_sum / np.maximum(m_count, 1))**2, 0.0)
m_val_std       = np.sqrt(m_var).astype(np.float32)
m_unique_users  = np.array([len(s) for s in m_users], dtype=np.float32)

m_repeat_u = np.zeros(merchantnum, dtype=np.float32)
for (uid, mid), cnt in edge_cnt.items():
    if cnt > 1:
        m_repeat_u[mid] += 1.0
m_user_repeat_rate = m_repeat_u / np.maximum(m_unique_users, 1.0)

# MCC → ordinal category index; missing MCC → len(unique_mccs) (unknown bucket for log_minmax)
unique_mccs = sorted(set(m_mcc_seen.values()))
mcc2idx = {mcc: i for i, mcc in enumerate(unique_mccs)}
_unknown_cat = float(len(unique_mccs))
m_cat = np.full(merchantnum, _unknown_cat, dtype=np.float32)
for mid, mcc in m_mcc_seen.items():
    m_cat[mid] = float(mcc2idx[mcc])

user_features = np.stack([
    u_avg_val,    # 0: avg_interaction_value  (quality signal)
    u_span_days,  # 1: activity_span_days     (temporal profile)
    u_recency,    # 2: recency_score          (temporal recency)
    u_val_std,    # 3: value_std_norm         (preference volatility)
], axis=1)

merchant_features = np.stack([
    m_avg_val,          # 0: avg_interaction_value  (quality signal)
    m_cat,              # 1: category_id (MCC index) (only exogenous feature)
    m_val_std,          # 2: value_std_norm          (price variance)
    m_user_repeat_rate, # 3: user_repeat_rate        (loyalty signal)
], axis=1)

print(f'User features     : {user_features.shape}')
print(f'Merchant features : {merchant_features.shape}')
print(f'Unique MCCs       : {len(unique_mccs):,}')
print(f'User   mean/col   : {user_features.mean(axis=0).round(4)}')
print(f'Merch  mean/col   : {merchant_features.mean(axis=0).round(4)}')

User features     : (1210, 4)
Merchant features : (17377, 4)
Unique MCCs       : 107
User   mean/col   : [6.834000e-01 7.262581e+02 9.959000e-01 5.650000e-02]
Merch  mean/col   : [6.92800e-01 7.01726e+01 2.71000e-02 8.58000e-01]


In [5]:
# ── Cell 5: Normalize 4+4 feature arrays ──────────────────────────────────────
# User  skip: col 0 (avg_value in (0,1)), col 2 (recency in [0,1])
# User  minmax-only: col 3 (value_std)
# User  log_minmax: col 1 (activity_span_days)
# Merch skip: col 0 (avg_value in (0,1)), col 3 (user_repeat_rate in [0,1])
# Merch minmax-only: col 2 (value_std)
# Merch log_minmax: col 1 (category_id — ordinal int index)

SKIP_NORM_USER  = {0, 2}  # avg_interaction_value, recency_score — already in [0,1]
SKIP_NORM_MERCH = {0, 3}  # avg_interaction_value, user_repeat_rate — already in [0,1]

def log_minmax(arr: np.ndarray) -> np.ndarray:
    v = np.log1p(np.clip(arr, 0, None))
    mn, mx = v.min(), v.max()
    return ((v - mn) / max(mx - mn, 1e-8)).astype(np.float32)

def minmax(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    return ((arr - mn) / max(mx - mn, 1e-8)).astype(np.float32)

user_features_norm  = user_features.copy()
merch_features_norm = merchant_features.copy()

for col in range(4):
    if col not in SKIP_NORM_USER:
        user_features_norm[:, col] = (
            minmax(user_features[:, col]) if col == 3       # value_std: minmax
            else log_minmax(user_features[:, col])          # activity_span: log_minmax
        )
for col in range(4):
    if col not in SKIP_NORM_MERCH:
        merch_features_norm[:, col] = (
            minmax(merchant_features[:, col]) if col == 2  # value_std: minmax
            else log_minmax(merchant_features[:, col])     # category_id: log_minmax
        )

print('After normalization:')
print(f'  User   min/max per col: {user_features_norm.min(axis=0).round(3)} / {user_features_norm.max(axis=0).round(3)}')
print(f'  Merch  min/max per col: {merch_features_norm.min(axis=0).round(3)} / {merch_features_norm.max(axis=0).round(3)}')

After normalization:
  User   min/max per col: [0.566 0.    0.159 0.   ] / [0.75 1.   1.   1.  ]
  Merch  min/max per col: [0.505 0.    0.    0.   ] / [0.86 1.   1.   1.  ]


In [6]:
# ── Cell 6: Edge re-weighting (log-quantile rank) ─────────────────────────────
# For amount-based data (finance, synthetic), the raw sigmoid(log1p/|·|/p75)
# transform (critic.md §3: narrow / skewed) is replaced with a monotone
# rank-percentile. This guarantees weights span [0,1] with uniform density
# independently of the amount distribution's tail, letting the graph
# normalisation step exploit full variance.
_pair_arr = np.array(list(edge_amt_sum.keys()), dtype=np.int64)
_cnt_arr  = np.array([edge_cnt[tuple(p)] for p in _pair_arr], dtype=np.float64)
_avg_amt  = np.array(
    [edge_amt_sum[tuple(p)] / max(edge_cnt[tuple(p)], 1)
     for p in _pair_arr], dtype=np.float64)

_log_amt  = np.log1p(_avg_amt)
_order    = np.argsort(np.argsort(_log_amt))
_N        = len(_order)
_w_norm   = (_order.astype(np.float64) + 1) / (_N + 1)

edge_weights: dict[tuple[int, int], float] = {
    (int(_pair_arr[i, 0]), int(_pair_arr[i, 1])): float(_w_norm[i])
    for i in range(_N)
}
vals = _w_norm
print(f'Edge weights (log-quantile rank): {len(edge_weights):,} pairs')
print(f'  log1p(avg_amt) range: [{_log_amt.min():.4f}, {_log_amt.max():.4f}]')
print(f'  norm stats  : min={vals.min():.4f} max={vals.max():.4f} '
      f'mean={vals.mean():.4f} std={vals.std():.4f}')
print(f'  percentiles : p05={np.percentile(vals,5):.4f}  p50={np.percentile(vals,50):.4f}  '
      f'p95={np.percentile(vals,95):.4f}')

Edge weights (log-quantile rank): 136,459 pairs
  log1p(avg_amt) range: [0.0000, 8.7970]
  norm stats  : min=0.0000 max=1.0000 mean=0.5000 std=0.2887
  percentiles : p05=0.0500  p50=0.5000  p95=0.9500


In [7]:
# ── Cell 7: Bipartite graph visualization ──────────────────────────────────────
# Geometric bipartite layout — users (left, blue circles) ↔ merchants (right, orange squares).
# Nodes are the top-N highest-degree users and their connected merchants.
# Edge thickness ∝ interaction weight (normalized to [0,1]).

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from collections import defaultdict as _dd

    np.random.seed(42)
    N_USR_SHOW = 20
    N_MRC_SHOW = 25

    # Index edge_weights by user
    u_to_m: dict = _dd(list)
    for (uid, mid), w in edge_weights.items():
        u_to_m[uid].append((mid, w))

    # Select highest-degree users
    top_users  = sorted(u_to_m.keys(), key=lambda u: len(u_to_m[u]), reverse=True)
    sample_u   = top_users[:N_USR_SHOW]

    # Gather connected merchants, capped at N_MRC_SHOW
    sample_m_set: set = set()
    for uid in sample_u:
        for mid, _ in u_to_m[uid]:
            sample_m_set.add(mid)
    sample_m    = sorted(sample_m_set)[:N_MRC_SHOW]
    m_visible   = set(sample_m)

    n_u, n_m = len(sample_u), len(sample_m)

    # Bipartite layout: users x=0, merchants x=1
    u_pos = {uid: (0.0, i / max(n_u - 1, 1)) for i, uid in enumerate(sample_u)}
    m_pos = {mid: (1.0, i / max(n_m - 1, 1)) for i, mid in enumerate(sample_m)}

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    ax.set_title(
        f'Bipartite Interaction Graph — {DATASET_NAME}\n'
        f'(top-{n_u} users by degree  ·  {n_m} connected merchants)',
        fontsize=13, fontweight='bold', pad=14,
    )

    visible_ws = [w for uid in sample_u for mid, w in u_to_m[uid] if mid in m_visible]
    w_max = max(visible_ws) if visible_ws else 1.0

    # Draw edges
    for uid in sample_u:
        for mid, w in u_to_m[uid]:
            if mid not in m_visible:
                continue
            p1, p2 = u_pos[uid], m_pos[mid]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                    color='#90A4AE', alpha=0.4,
                    linewidth=0.4 + 2.2 * (w / w_max), zorder=1)

    # User nodes (blue circles)
    ax.scatter([u_pos[u][0] for u in sample_u], [u_pos[u][1] for u in sample_u],
               s=220, c='#1565C0', zorder=4, edgecolors='white', linewidths=1.8, label='User')
    for uid, (x, y) in u_pos.items():
        ax.text(x - 0.05, y, f'u{uid}', ha='right', va='center', fontsize=7, color='#1565C0')

    # Merchant nodes (orange squares)
    ax.scatter([m_pos[m][0] for m in sample_m], [m_pos[m][1] for m in sample_m],
               s=220, c='#E65100', marker='s', zorder=4, edgecolors='white', linewidths=1.8, label='Merchant')
    for mid, (x, y) in m_pos.items():
        ax.text(x + 0.05, y, f'm{mid}', ha='left', va='center', fontsize=7, color='#E65100')

    ax.text(0.0, -0.08, f'Users  (n={n_u})',
            ha='center', fontsize=11, color='#1565C0', fontweight='bold',
            transform=ax.transData)
    ax.text(1.0, -0.08, f'Merchants  (n={n_m})',
            ha='center', fontsize=11, color='#E65100', fontweight='bold',
            transform=ax.transData)

    ax.set_xlim(-0.22, 1.22)
    ax.set_ylim(-0.12, 1.08)
    ax.legend(loc='upper center', ncol=2, fontsize=9, framealpha=0.8)

    plt.tight_layout()
    plt.savefig(OUT_DIR + 'bipartite_graph_sample.png', bbox_inches='tight', dpi=120)
    plt.show()
    print(f'Saved: bipartite_graph_sample.png  ({n_u} users, {n_m} merchants, {len(visible_ws)} edges shown)')

except Exception as e:
    print(f'Graph visualization skipped: {e}')

Saved: bipartite_graph_sample.png  (20 users, 25 merchants, 30 edges shown)


C:\Users\User\AppData\Local\Temp\ipykernel_31772\2872773007.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── Cell 8: Save all files + summary ───────────────────────────────────────────
np.save(OUT_DIR + 'user_features.npy',     user_features_norm)
np.save(OUT_DIR + 'merchant_features.npy', merch_features_norm)

with open(OUT_DIR + 'edge_weights.pkl', 'wb') as f:
    pickle.dump(edge_weights, f)

meta = {
    'user_feature_names': [
        'avg_interaction_value',   # col 0
        'activity_span_days',      # col 1
        'recency_score',           # col 2
        'value_std_norm',          # col 3
    ],
    'merchant_feature_names': [
        'avg_interaction_value',   # col 0
        'category_id',             # col 1
        'value_std_norm',          # col 2
        'user_repeat_rate',        # col 3
    ],
    'edge_feature_names':  ['log_quantile_rank_amount'],
    'user_value_col':      0,   # avg_interaction_value is col 0 in 4+4 schema
    'merchant_value_col':  0,   # avg_interaction_value is col 0 in 4+4 schema
    'edge_reweight_strategy': 'log_quantile_rank',
    'edge_weight_range':   [float(vals.min()), float(vals.max())],
    'dataset':             DATASET_NAME,
    'schema':              '4+4',
    'mcc_unique_count':    len(unique_mccs),
    'feature_selection':   'dropped graph-redundant: interaction_count, unique_user_count, txns_per_user, popularity_rank',
    'window_last_n_years': LAST_N_YEARS,
    'window_cutoff_ts':    CUTOFF_TS,
    'window_max_ts':       MAX_TS,
}
with open(OUT_DIR + 'feature_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fname in ['user_features.npy', 'merchant_features.npy', 'edge_weights.pkl', 'feature_meta.json']:
    path = OUT_DIR + fname
    size = os.path.getsize(path) if os.path.isfile(path) else None
    status = f'{size/1024:.0f} KB' if size else 'MISSING'
    print(f'  {fname:<35} {status}')
print()
print('=' * 60)
print(f'Feature Extraction Complete — {DATASET_NAME}  [4+4 schema]')
print('=' * 60)
print(f'User feature shape     : {user_features_norm.shape}')
print(f'Merchant feature shape : {merch_features_norm.shape}')
print(f'Edge weights           : {len(edge_weights):,} pairs')
print(f'Schema (4+4)           : user_dim={user_features_norm.shape[1]}, merch_dim={merch_features_norm.shape[1]}')

Saved:
  user_features.npy                   19 KB
  merchant_features.npy               272 KB
  edge_weights.pkl                    2237 KB
  feature_meta.json                   1 KB

Feature Extraction Complete — finance-merchant  [4+4 schema]
User feature shape     : (1210, 4)
Merchant feature shape : (17377, 4)
Edge weights           : 136,459 pairs
Schema (4+4)           : user_dim=4, merch_dim=4
